In [2]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

RAW_KG_PATH = "kg.csv"
OUTPUT_CSV_PATH = "NACC_Seed_DementiaHKG.csv"

def build_seed_and_export_csv_nacc():
    print("==================================================")
    print("🕵️ 步骤 1: 加载 PrimeKG 数据库...")
    try:
        df = pd.read_csv(RAW_KG_PATH, low_memory=False)
        all_nodes = set(df['x_name'].dropna()).union(set(df['y_name'].dropna()))
        print(f"   ✅ 成功加载 {RAW_KG_PATH}，共提取独立节点 {len(all_nodes)} 个。")
    except FileNotFoundError:
        print(f"   ❌ 找不到 {RAW_KG_PATH}，请检查路径。")
        return

    print("\n==================================================")
    print("🧬 步骤 2: 初始化医学共识核心种子 (Core Seeds)...")
    
    core_seeds = {
        "Alzheimer disease": "Core_Disease", 
        "Dementia": "Core_Disease",
        "Cognitive impairment": "Core_Disease",
        "APOE": "Core_Gene",
        "MAPT": "Core_Gene",   
        "APP": "Core_Gene",    
        "PSEN1": "Core_Gene",
        "PSEN2": "Core_Gene",
        "TREM2": "Core_Gene"
    }

    print("\n==================================================")
    print("📊 步骤 3: 补全 NACC 临床与病史表型映射字典...")
    
    nacc_mapping = {
        # --- NPI-Q 精神神经症状 ---
        "npiq_DEL": "Delusions",
        "npiq_HALL": "Hallucinations",
        "npiq_AGIT": "Agitation",
        "npiq_DEPD": "Depressivity",          
        "npiq_ANX": "Anxiety",
        "npiq_ELAT": "Euphoria",
        "npiq_APA": "Apathy",
        "npiq_DISN": "Disinhibition",
        "npiq_IRR": "Irritability",
        "npiq_MOT": "Restlessness",
        "npiq_NITE": "Sleep disturbance",     
        "npiq_APP": "Poor appetite",          

        # --- 认知与功能量表 ---
        "lm_imm": "Memory impairment",     
        "lm_del": "Memory impairment",
        "craft_imm": "Memory impairment",
        "craft_del": "Memory impairment",
        "boston": "Aphasia",               
        "mint": "Aphasia",
        "animal": "Aphasia",               
        "Fwords": "Aphasia",
        "trailA": "Impaired executive functioning", 
        "trailB": "Impaired executive functioning", 
        "digitB": "Short attention span",           
        "digitBL": "Short attention span",          
        "digitF": "Short attention span",           
        "digitFL": "Short attention span",
        "numberB": "Short attention span",
        "numberBL": "Short attention span",
        "numberF": "Short attention span",
        "numberFL": "Short attention span",
        "faq_BILLS": "Impaired executive functioning", 
        "faq_TAXES": "Impaired executive functioning", 
        "faq_SHOPPING": "Memory impairment",
        "faq_REMDATES": "Memory impairment",
        "faq_GAMES": "Impaired executive functioning",
        "faq_STOVE": "Memory impairment",
        "faq_MEALPREP": "Impaired executive functioning",
        "faq_EVENTS": "Short attention span",
        "faq_PAYATTN": "Short attention span",
        "faq_TRAVEL": "Impaired executive functioning",
        "gds": "Depressivity",                      

        # --- EHR 既往病史映射 (全量精确命中版) ---
        "his_CVHATT": "myocardial infarction", 
        "his_CVAFIB": "atrial fibrillation (disease)",
        "his_CVCHF": "heart failure",
        "his_CBSTROKE": "cerebral infarction", 
        "his_CBTIA": "transient ischemic attack (disease)",
        "his_SEIZURES": "Seizure",
        "his_HYPERTEN": "hypertension",        
        "his_HYPERCHO": "Hypercholesterolemia",
        "his_DIABETES": "type 2 diabetes mellitus",
        "his_THYROID": "hypothyroidism",
        "his_B12DEF": "vitamin B12 deficiency",
        "his_PSYCDIS": "mental disorder",      
        "his_OCD": "obsessive-compulsive disorder",
        "his_ANXIETY": "Anxiety",
        "his_SCHIZ": "schizophrenia",
        "his_BIPOLAR": "bipolar disorder",
        "his_PTSD": "post-traumatic stress disorder",
        "his_ALCOHOL": "Alcoholism",
        "his_DEP2YRS": "Depressivity",
        "his_DEPOTHR": "Depressivity"          
    }

    print("\n==================================================")
    print("🔍 步骤 4: 在 PrimeKG 中校验节点有效性...")
    
    valid_records = []
    
    for seed, category in core_seeds.items():
        if seed in all_nodes:
            valid_records.append({"NACC_Feature": "Core_Prior_Knowledge", "PrimeKG_Node": seed, "Category": category})
            print(f"     ✅ 命中: {seed}")
        else:
            print(f"     ❌ 未命中: {seed}")

    for nacc_feat, pkg_node in nacc_mapping.items():
        if pkg_node in all_nodes:
            valid_records.append({"NACC_Feature": nacc_feat, "PrimeKG_Node": pkg_node, "Category": "Clinical_Mapping"})
            print(f"     ✅ 命中: {nacc_feat} -> {pkg_node}")
        else:
            print(f"     ❌ 未命中: {nacc_feat} -> {pkg_node}")

    print("\n==================================================")
    print("🚫 步骤 5: 附加全局黑名单规则...")
    for rule in ["drug", "exposure", "indication", "contraindication", "off-label use"]:
        cat = "BLACKLIST_TYPE" if rule in ["drug", "exposure"] else "BLACKLIST_REL"
        valid_records.append({"NACC_Feature": cat, "PrimeKG_Node": rule, "Category": "Rule"})

    print("\n==================================================")
    print("💾 步骤 6: 导出为 CSV 文件...")
    out_df = pd.DataFrame(valid_records)
    out_df.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8')
    print(f"   ✅ 导出完成！文件已保存至: {OUTPUT_CSV_PATH}")

if __name__ == "__main__":
    build_seed_and_export_csv_nacc()

🕵️ 步骤 1: 加载 PrimeKG 数据库...
   ✅ 成功加载 kg.csv，共提取独立节点 129262 个。

🧬 步骤 2: 初始化医学共识核心种子 (Core Seeds)...

📊 步骤 3: 补全 NACC 临床与病史表型映射字典...

🔍 步骤 4: 在 PrimeKG 中校验节点有效性...
     ✅ 命中: Alzheimer disease
     ✅ 命中: Dementia
     ✅ 命中: Cognitive impairment
     ✅ 命中: APOE
     ✅ 命中: MAPT
     ✅ 命中: APP
     ✅ 命中: PSEN1
     ✅ 命中: PSEN2
     ✅ 命中: TREM2
     ✅ 命中: npiq_DEL -> Delusions
     ✅ 命中: npiq_HALL -> Hallucinations
     ✅ 命中: npiq_AGIT -> Agitation
     ✅ 命中: npiq_DEPD -> Depressivity
     ✅ 命中: npiq_ANX -> Anxiety
     ✅ 命中: npiq_ELAT -> Euphoria
     ✅ 命中: npiq_APA -> Apathy
     ✅ 命中: npiq_DISN -> Disinhibition
     ✅ 命中: npiq_IRR -> Irritability
     ✅ 命中: npiq_MOT -> Restlessness
     ✅ 命中: npiq_NITE -> Sleep disturbance
     ✅ 命中: npiq_APP -> Poor appetite
     ✅ 命中: lm_imm -> Memory impairment
     ✅ 命中: lm_del -> Memory impairment
     ✅ 命中: craft_imm -> Memory impairment
     ✅ 命中: craft_del -> Memory impairment
     ✅ 命中: boston -> Aphasia
     ✅ 命中: mint -> Aphasia
     ✅ 命中: anim